### Sampling initial conditions
Import libraries

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from thermos import *
from sampling_fun import *

Set inputs

In [ ]:
# Reading data
path = os.path.abspath(os.getcwd() + f"/../pde/thermolib/strang/output/")
cols = ['rho_O','rho_O2','u','Th','Tv']
# Sampling data
bounds_raw = {
    'XO2': [  0.,      1.],
    'p':   [900., 100000.],
    'Th':  [300.,  10000.],
    'Tv':  [300.,  10000.]
}
nb_train_cases = 5120
nb_test_cases  = 50
kde_samples_perc = 0.025
extra_perc=0.1
# Saving path
saving_path = os.path.abspath(os.getcwd() + f"/outputs/sampling/")
if os.path.exists(saving_path):
    shutil.rmtree(saving_path)
os.makedirs(saving_path, exist_ok=True)

Read solution data

In [ ]:
data = []
for file in os.listdir(path):
    if file.startswith('restart_EU_NLTE') and file.endswith('_0.dat') and file != 'restart_EU_NLTE_0.dat':
        filename = path + '/' + file
        data.append(np.loadtxt(filename))
data = np.vstack(data)
df = pd.DataFrame(data, columns=cols)

Initialize thermodynamics and extract sampling variables: $X = [X_{O_2}, p, T_h, T_v, T_h-T_v]$

In [ ]:
sp_th, mix = init_thermos()

In [ ]:
X = extract_sampling_qoi(df, sp_th, mix)

Set bounds

In [ ]:
# Compute bounds
phys_bounds = np.array(list(bounds_raw.values())).T
bounds = compute_bounds(
    X,
    phys_bounds=phys_bounds,
    extra_perc=extra_perc
)
# Correct max dT
bounds[-1,-1] = bounds[1,2] - bounds[0,2]

In [ ]:
# Saving computed bounds
df = pd.DataFrame(bounds, columns=list(bounds_raw.keys())+['dT'])
df.index = ['Min', 'Max']
df.to_csv(saving_path + '/bounds.csv', float_format='%.2f')
print('Defined bounds:')
df

Construct kde

In [ ]:
X_ = np.take(X, np.arange(X.shape[1]-1), 1)
scaler = construct_scaler(X_)
kde = construct_kde(X_, bandwidth=0.05, scaler=scaler)

Uniform and kde-based sampling

In [ ]:
def check_samples(samples, bounds):
    n = len(samples)
    dT = -np.diff(np.take(samples, [-2,-1], axis=1))
    samples = np.hstack([samples, dT])
    cond = (samples >= bounds[0]) * (samples <= bounds[1])
    cond = np.product(cond, axis=1)
    idx  = np.where(cond)[0]
    samples = samples[idx]
    return samples, n - len(idx)

In [ ]:
nb_samples_uniform = int((nb_train_cases*2 + nb_test_cases)*(1-kde_samples_perc))
nb_samples_kde = (nb_train_cases*2 + nb_test_cases) - nb_samples_uniform
bounds_prim = np.take(bounds, np.arange(bounds.shape[1]-1), axis=1)

# Uniform
samples = []
n = nb_samples_uniform
while n > 0:
    samples_n = sample_space(
        bounds=bounds_prim,
        nb_samples=n
    )
    samples_n, n = check_samples(samples_n, bounds)
    samples.append(samples_n)
samples_uniform = np.vstack(samples)

# KDE
samples = []
n = nb_samples_kde
while n > 0:
    samples_n = sample_kde(
        kde,
        nb_samples=n,
        scaler=scaler,
        bounds=bounds_prim
    )
    samples_n, n = check_samples(samples_n, bounds)
    samples.append(samples_n)
samples_kde = np.vstack(samples)

samples = np.vstack([samples_uniform, samples_kde])
samples = np.take(samples, np.arange(len(bounds_raw)), 1)

Save samples in files

In [ ]:
# Getting indices
# Testing samples
indices = np.arange(len(samples))
np.random.shuffle(indices)
idx_test = np.random.choice(indices, size=nb_test_cases, replace=False)
# Training samples
idx_train = np.setdiff1d(np.arange(len(samples)), idx_test)
assert np.intersect1d(idx_test, idx_train).size == 0
# Testing samples for checking training
idx_testing_train = np.random.choice(idx_train, size=nb_test_cases, replace=False)
idx_test = np.concatenate([idx_test, idx_testing_train])

In [ ]:
# Saving
# Testing samples
for (k, idx) in (('test', idx_test),('train_valid', idx_train)):
    df = pd.DataFrame(samples[idx], columns=list(bounds_raw.keys()))
    path_samples = saving_path + f'/data/{k}/'
    os.makedirs(path_samples, exist_ok=True)
    df.to_csv(path_samples + '/inp_space.csv', float_format='%.8e', index=False)

Visualize samples

In [ ]:
path_figs = saving_path + '/figs/'
os.makedirs(path_figs, exist_ok=True)

labels = ["$T_v$ [K]", "$T_h$ [K]"]
Th, Tv = samples[:,2], samples[:,3]

plot_hist_3d(Tv, Th, labels, figname=path_figs+'/hist_3d.pdf', nb_bins=40, show=False)
plot_hist_2d(Th, Tv, np.flip(labels), figname=path_figs+'/hist_2d.pdf', nb_bins=40, show=False)